In [40]:
#Importing libraries
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import joblib

In [41]:
#Loading the dataset
df = pd.read_csv("C:\\Faisal env\\Faisal_env\\MscAI&ML\\Projects\\IDSS_Major_Project\\Dataset\\walmart_processed_data.csv")

In [42]:
#loading the models and data splits
lr_predictions, lr_pipeline = joblib.load("C:\\Faisal env\\Faisal_env\\MscAI&ML\\Projects\\IDSS_Major_Project\\Joblib\\walmart_lr.joblib")
rf_model, rf_predictions = joblib.load("C:\\Faisal env\\Faisal_env\\MscAI&ML\\Projects\\IDSS_Major_Project\\Joblib\\walmart_rf.joblib")
xgb_model, xgb_predictions = joblib.load("C:\\Faisal env\\Faisal_env\\MscAI&ML\\Projects\\IDSS_Major_Project\\Joblib\\walmart_xgb.joblib")

In [43]:
#loading the data splits
train_df, test_df = joblib.load("C:\\Faisal env\\Faisal_env\\MscAI&ML\\Projects\\IDSS_Major_Project\\Joblib\\data_frame.joblib")
X_train, y_train, X_test, y_test = joblib.load("C:\\Faisal env\\Faisal_env\\MscAI&ML\\Projects\\IDSS_Major_Project\\Joblib\\data_splits.joblib")
print("X_train shape:", X_train.shape)

X_train shape: (332778, 26)


Inventory Optimization

In [44]:
#IDSS Implementation
lead_time = 1

In [45]:
#Creating a DataFrame to hold forecasts and decisions
forecasts = test_df[["Store", "Dept", "Date"]].copy()
forecasts["Forecast_Sales"] = xgb_predictions

In [46]:
#Calculating average weekly sales for each Store Dept combination
avg_sales = df.groupby(["Store", "Dept"])["Weekly_Sales"].mean().reset_index()
avg_sales.rename(columns={"Weekly_Sales": "Avg_Weekly_Sales"}, inplace=True)

In [47]:
#Generating inventory decisions based on forecasts and current inventory
forecasts = forecasts.merge(avg_sales, on=["Store", "Dept"], how="left")
forecasts["Current_Inventory"] = forecasts["Avg_Weekly_Sales"] * 4

In [48]:
#Calculating safety stock and reorder points
forecasts["Safety_Stock"] = (forecasts["Forecast_Sales"] * 0.15).round(0)
forecasts["Reorder_Point"] = (forecasts["Forecast_Sales"] * lead_time) + forecasts["Safety_Stock"]
forecasts

,Store,Dept,Date,Forecast_Sales,Avg_Weekly_Sales,Current_Inventory,Safety_Stock,Reorder_Point
0,1,1,2012-04-06,29394.105469,22513.322937,90053.291748,4409.0,33803.105469
1,1,1,2012-04-13,36061.191406,22513.322937,90053.291748,5409.0,41470.191406
2,1,1,2012-04-20,46537.578125,22513.322937,90053.291748,6981.0,53518.578125
3,1,1,2012-04-27,24665.898438,22513.322937,90053.291748,3700.0,28365.898438
4,1,1,2012-05-04,19050.939453,22513.322937,90053.291748,2858.0,21908.939453
...,...,...,...,...,...,...,...,...
88787,45,98,2012-09-28,185.126801,561.239037,2244.956148,28.0,213.126801
88788,45,98,2012-10-05,627.267761,561.239037,2244.956148,94.0,721.267761
88789,45,98,2012-10-12,345.078949,561.239037,2244.956148,52.0,397.078949
88790,45,98,2012-10-19,747.276978,561.239037,2244.956148,112.0,859.276978


In [49]:
#Generate inventory decisions
def dss_engine(row):
    if row["Current_Inventory"] < row["Reorder_Point"]:
        return "Stockout Risk"

    elif row["Forecast_Sales"] > row["Avg_Weekly_Sales"] * 1.3:
        return "High Demand"

    elif row["Forecast_Sales"] < row["Avg_Weekly_Sales"] * 0.7:
        return "Low Demand"

    else:
        return "Optimal Stock"
    
forecasts["Final_DSS_Alert"] = forecasts.apply(dss_engine, axis=1)
print("DSS Alert Engine Summary:")
print(forecasts["Final_DSS_Alert"].value_counts())

DSS Alert Engine Summary:
Final_DSS_Alert
Optimal Stock    66793
Low Demand       12837
High Demand       8126
Stockout Risk     1036
Name: count, dtype: int64


In [50]:
# Manager's Action Report Generation
alerts = []
for _, row in forecasts.iterrows():
    action = "No action required"
    if row["Final_DSS_Alert"] == "Stockout Risk":
        action = f"Order {max(0, int(row['Reorder_Point'] - row['Current_Inventory']))} units immediately"
    elif row["Final_DSS_Alert"] == "High Demand":
        action = "Increase buffer stock by 20%, monitor trends daily"
    elif row["Final_DSS_Alert"] == "Low Demand":
        action = "Reduce upcoming procurement orders to save costs"

    alerts.append({
        'Store': row['Store'],
        'Dept': row['Dept'],
        'Date': row['Date'].strftime('%Y-%m-%d') if hasattr(row['Date'], 'strftime') else str(row['Date']),
        'Forecast': f"${row['Forecast_Sales']:,.0f}",
        'Inventory': f"${row['Current_Inventory']:,.0f}",
        'Priority': row['Final_DSS_Alert'],
        'Action': action
    })

alert_df = pd.DataFrame(alerts)
print(f"Total Actionable Alerts Generated: {len(alert_df)}")

Total Actionable Alerts Generated: 88792


In [51]:
#Main processed data
df.to_csv("walmart_sales.csv", index=False)

#Forecast results
forecasts[["Store","Dept","Date","Avg_Weekly_Sales","Forecast_Sales",
           "Current_Inventory", "Reorder_Point","Final_DSS_Alert"]].to_csv("walmart_forecasts.csv", index=False)

#Manager's Action Alert report
alert_df.to_csv("walmart_alerts.csv", index=False)

print("All final CSV files have been successfully exported")

All final CSV files have been successfully exported
